In [ ]:
import json
import os
import re
import urllib.parse as up
import lightning
import matplotlib.pyplot as plt
import numpy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.metrics import confusion_matrix, roc_curve, auc
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from torchmetrics import MetricCollection
from torchmetrics.classification import BinaryAccuracy, BinaryPrecision, BinaryRecall, BinaryF1Score
from url_dataset import UrlDataset, collate_fn

n_workers = min(4, os.cpu_count() - 1)

torch.set_float32_matmul_precision('medium')

SAFE_CHARS = "-._~:/?#[]@!$&'()*+,;=%"
ALLOWED = set("abcdefghijklmnopqrstuvwxyz0123456789" + SAFE_CHARS)


def clean_url(raw: str) -> str:
    raw = raw.strip()
    if not re.match(r'^[a-z]+://', raw):
        raw = "http://" + raw
    u = up.unquote(raw).lower()
    u = re.sub(r'^(https?://)www\.', r'\1', u)
    u = re.sub(r':(80|443)(?=/)', '', u)
    if u.endswith('/') and u.count('/') > 2:
        u = u[:-1]
    u = ''.join(c if c in ALLOWED else '_' for c in u)

    return u

In [ ]:
email_df = pd.read_csv("dataset/urls/new_data_urls.csv")
email_df['label'] = 1 - email_df['status']
email_df.drop_duplicates(['url'], inplace=True)
email_df_phishing = email_df[email_df['label'] == 1]
email_df_legitime = email_df[email_df['label'] == 0][:381014]
email_df = email_df.drop_duplicates(subset=['url'])
email_df = pd.concat([email_df_legitime, email_df_phishing], ignore_index=True)

In [ ]:
del email_df_phishing
del email_df_legitime
email_df["url"] = [clean_url(url) for url in email_df["url"]]

In [ ]:
CHARS = list(ALLOWED)
PAD, UNK = 0, 1
MAX_LEN = 200

if os.path.exists("url_data/url_char_vocab.json"):
    with open("url_data/url_char_vocab.json", "r") as f:
        char2idx = json.load(f)
else:
    char2idx = {c: i + 2 for i, c in enumerate(CHARS)}
    with open("url_data/url_char_vocab.json", "w") as fw:
        json.dump(char2idx, fw, ensure_ascii=False, indent=2)


def encode_url(url: str) -> list:
    ids = [char2idx.get(c, UNK) for c in url[:MAX_LEN]]
    return ids + [PAD] * (MAX_LEN - len(ids))

In [ ]:
email_df["encoded"] = [encode_url(url) for url in email_df["url"]]

In [ ]:
email_df.drop(columns=["url", "status"], inplace=True)

In [ ]:
X = email_df['encoded'].tolist()
y = email_df['label'].tolist()

del email_df

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

train_ds = UrlDataset(X_train, y_train, max_length=MAX_LEN)
val_ds = UrlDataset(X_val, y_val, max_length=MAX_LEN)
test_ds = UrlDataset(X_test, y_test, max_length=MAX_LEN)

batch_size = 128
loader_kwargs = dict(
    batch_size=batch_size,
    collate_fn=collate_fn,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=1
)

train_loader = DataLoader(
    train_ds,
    shuffle=True,
    drop_last=True,
    **loader_kwargs
)
val_loader = DataLoader(
    val_ds,
    shuffle=False,
    drop_last=False,
    **loader_kwargs
)
test_loader = DataLoader(
    test_ds,
    shuffle=False,
    drop_last=False,
    **loader_kwargs
)

In [ ]:
class AttentionPool(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.attn = nn.Linear(dim, 1)

    def forward(self, x, mask=None):
        scores = self.attn(x).squeeze(-1)

        if mask is not None:
            scores = scores.masked_fill(~mask, float("-inf"))

        weights = torch.softmax(scores, dim=1)
        pooled = torch.sum(x * weights.unsqueeze(-1), dim=1)
        return pooled

In [ ]:
class Phishign_URL_CNN(lightning.LightningModule):
    def __init__(self, emb_dim=128, n_filters=128, dropout=0.3, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()
        self.lr = lr
        vocab_size = len(char2idx) + 2
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)

        conv_layers = []
        in_ch = emb_dim
        k = 3
        for _ in range(3):
            conv_layers += [
                nn.Conv1d(in_ch, n_filters, k, padding=k//2),
                nn.LeakyReLU(),
                nn.BatchNorm1d(n_filters)
            ]
            in_ch = n_filters
            k += 2
        self.conv = nn.Sequential(*conv_layers)
        self.attn_pool = AttentionPool(n_filters)
        self.pool = nn.AdaptiveMaxPool1d(1)
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(2 * n_filters, 1)
        )

        self.crit = nn.BCEWithLogitsLoss()

        base_metrics = MetricCollection({
            "acc": BinaryAccuracy(),
            "prec": BinaryPrecision(),
            "rec": BinaryRecall(),
            "f1": BinaryF1Score()
        })

        self.train_metrics = base_metrics.clone(prefix="train_")
        self.val_metrics = base_metrics.clone(prefix="val_")
        self.test_metrics = base_metrics.clone(prefix="test_")

    def forward(self, x):
        mask = (x != PAD)

        x = self.emb(x).transpose(1, 2)
        x = self.conv(x)

        max_pooled = self.pool(x).squeeze(-1)

        attn_input = x.transpose(1, 2)
        attn_pooled = self.attn_pool(attn_input, mask)

        x = torch.cat([max_pooled, attn_pooled], dim=1)

        return self.head(x).squeeze(-1)

    def _shared_step(self, batch, stage):
        x, y = batch
        logits = self(x)
        loss = self.crit(logits, y.float())

        preds = torch.sigmoid(logits)
        metric_set = getattr(self, f"{stage}_metrics")

        metric_set.update(preds, y.long())
        self.log(f"{stage}_loss", loss, prog_bar=True, on_epoch=True, on_step=False)
        self.log_dict(metric_set, prog_bar=True, on_epoch=True, on_step=False)

        return loss

    def training_step(self, batch, batch_idx):
        return self._shared_step(batch, "train")

    def validation_step(self, batch, batch_idx):
        self._shared_step(batch, "val")

    def test_step(self, batch, batch_idx):
        self._shared_step(batch, "test")

    def on_train_epoch_end(self):
        self.train_metrics.reset()

    def on_validation_epoch_end(self):
        self.val_metrics.reset()

    def on_test_epoch_end(self):
        self.test_metrics.reset()

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(),
                                lr=self.lr,
                                weight_decay=1e-4)
        sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt,
                                                         mode="min",
                                                         patience=3)
        return {
            "optimizer": opt,
            "lr_scheduler": {
                "scheduler": sch,
                "monitor": "val_loss"
            }
        }


In [ ]:
model = Phishign_URL_CNN(lr=1e-4)

es = EarlyStopping(monitor="val_loss", patience=3, mode="min")
ckpt = ModelCheckpoint(monitor="val_f1",
                       mode="max",
                       filename="best-phishing-url",
                       save_top_k=1)

csv_logger = CSVLogger("lightning_logs", name="phishing-url")

trainer = Trainer(
    max_epochs=50,
    accelerator="gpu",
    devices=1,
    deterministic=False,
    callbacks=[es, ckpt],
    log_every_n_steps=50,
    gradient_clip_val=1.0,
    logger=csv_logger,
    num_sanity_val_steps=0
)
trainer.fit(model, train_loader, val_loader)

trainer.test(ckpt_path=ckpt.best_model_path, dataloaders=test_loader)

In [ ]:
model = Phishign_URL_CNN.load_from_checkpoint(ckpt.best_model_path, weights_only=False)

In [ ]:
def plot_confusion_matrix(y_true, y_pred, labels=None, normalize=True, cmap="Blues"):
    cm = confusion_matrix(
        y_true, y_pred, normalize="true" if normalize else None
    )
    fig, ax = plt.subplots(figsize=(4, 4))
    im = ax.imshow(cm, interpolation="nearest", cmap=cmap, vmin=0, vmax=1 if normalize else None)
    ax.set(title="Confusion Matrix",
           xlabel="Predicted label", ylabel="True label")

    if labels is None:
        labels = ["0", "1"]
    ax.set_xticks(range(len(labels)), labels)
    ax.set_yticks(range(len(labels)), labels)

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            val = f"{cm[i, j]:.2f}" if normalize else f"{cm[i, j]:d}"
            color = "white" if cm[i, j] > (0.5 if normalize else cm.max() / 2) else "black"
            ax.text(j, i, val, ha="center", va="center", color=color, fontsize=10)

    fig.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout()
    plt.show()

    return cm


def plot_roc_auc(y_true, y_scores, threshold=0.5):
    fpr, tpr, thresh = roc_curve(y_true, y_scores)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(4, 4))
    plt.plot(fpr, tpr, lw=2, label=f"AUC = {roc_auc:.4f}")
    plt.plot([0, 1], [0, 1], linestyle="--", color="gray")

    idx = numpy.argmin(numpy.abs(thresh - threshold))
    plt.scatter(fpr[idx], tpr[idx], marker="o", color="red",
                label=f"thr={threshold:.2f}")

    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve")
    plt.legend()
    plt.tight_layout()
    plt.show()

    return roc_auc

In [ ]:
def plot_train_and_validation_metrics(path: str):
    data = pd.read_csv(path)
    train_data = data[1::2]
    val_data = data[0::2]

    plt.plot(train_data["train_acc"].tolist(), linestyle='-', label="Training Accuracy", color="red")
    plt.plot(val_data["val_acc"].tolist(), linestyle='-', label="Validation Accuracy", color="blue")

    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.show()

    plt.plot(train_data["train_loss"].tolist(), linestyle='-', label="Training Loss", color="red")
    plt.plot(val_data["val_loss"].tolist(), linestyle='-', label="Validation Loss", color="blue")

    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()

plot_train_and_validation_metrics("lightning_logs/phishing-url/version_0/metrics.csv")

In [ ]:
y_true, y_pred, y_scores = [], [], []
model.eval()
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(model.device)
        logits = model(xb).cpu()
        probs = torch.sigmoid(logits)

        y_true.extend(yb.cpu().tolist())
        y_pred.extend((probs >= 0.5).long().tolist())
        y_scores.extend(probs.tolist())

cm = plot_confusion_matrix(y_true, y_pred, normalize=False, labels=['legitimate', 'phishing'])
plot_roc_auc(y_true, y_scores)

In [ ]:
def bootstrap_ci(y_true, y_scores, thr=0.5, n_boot=1000, alpha=0.05, seed=42):
    rng = np.random.default_rng(seed)
    metrics = {"acc": [], "prec": [], "rec": [], "f1": [], "auc": []}

    y_true = np.asarray(y_true)
    y_scores = np.asarray(y_scores)

    for _ in range(n_boot):
        idx = rng.integers(0, len(y_true), len(y_true))
        yt, ys = y_true[idx], y_scores[idx]
        yp = (ys >= thr).astype(int)

        metrics["acc"].append(accuracy_score(yt, yp))
        metrics["prec"].append(precision_score(yt, yp, zero_division=0))
        metrics["rec"].append(recall_score(yt, yp))
        metrics["f1"].append(f1_score(yt, yp))
        metrics["auc"].append(roc_auc_score(yt, ys))

    ci = {
        m: (np.percentile(vals, 100 * alpha / 2),
            np.percentile(vals, 100 * (1 - alpha / 2)))
        for m, vals in metrics.items()
    }
    return ci


ci95 = bootstrap_ci(y_true, y_scores, thr=0.5)

for m, (lo, hi) in ci95.items():
    print(f"{m.upper():5s} 95% CI: {lo:.4f} – {hi:.4f}")

In [ ]:
import torch
from time import perf_counter

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device).eval()

tok_ids = torch.tensor(X_test[0]).to(device)
length = torch.tensor([tok_ids.numel()]).to(device)

tok_ids = tok_ids.unsqueeze(0)

N, times = 100, []
with torch.inference_mode():
    for _ in range(N):
        if device == "cuda":
            torch.cuda.synchronize()
        t0 = perf_counter()

        _ = model(tok_ids)

        if device == "cuda":
            torch.cuda.synchronize()
        times.append(perf_counter() - t0)

mean = sum(times) / N * 1000
std = torch.std(torch.tensor(times)) * 1000
print(f"Medie: {mean:.2f} ms  |  Dev. std: {std:.2f} ms")